In [1]:
%pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import ssl
import certifi
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

import copy
import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms, models

# -----------------------
# 1. Reproducibility
# -----------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# -----------------------
# 2. Paths + device
# -----------------------
BASE_DIR = Path("../../data/dataset/archive/original")
print("Dataset path:", BASE_DIR.resolve())

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# -----------------------
# 3. Transforms (train + val)
# -----------------------
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomRotation(12),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# -----------------------
# 4. Build stratified train/val split
# -----------------------
base_dataset = datasets.ImageFolder(root=str(BASE_DIR), transform=None)
class_names = base_dataset.classes
num_classes = len(class_names)
targets = base_dataset.targets

print("Classes:", class_names)
print("Num classes:", num_classes)

class_counts = Counter(targets)
for idx, name in enumerate(class_names):
    print(f"{name:10s}: {class_counts[idx]}")

train_indices = []
val_indices = []
rng = random.Random(SEED)

for class_idx in range(num_classes):
    idxs = [i for i, t in enumerate(targets) if t == class_idx]
    rng.shuffle(idxs)
    n_val = max(1, int(0.2 * len(idxs)))
    val_indices.extend(idxs[:n_val])
    train_indices.extend(idxs[n_val:])

print(f"Train samples: {len(train_indices)}")
print(f"Val samples:   {len(val_indices)}")

# Create two dataset objects so each split uses its own transform
train_dataset_all = datasets.ImageFolder(root=str(BASE_DIR), transform=train_transform)
val_dataset_all = datasets.ImageFolder(root=str(BASE_DIR), transform=val_transform)

train_dataset = Subset(train_dataset_all, train_indices)
val_dataset = Subset(val_dataset_all, val_indices)

# Weighted sampler to reduce class bias during training
train_class_counts = Counter(targets[i] for i in train_indices)
sample_weights = [1.0 / train_class_counts[targets[i]] for i in train_indices]
train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
 )

train_loader = DataLoader(train_dataset, batch_size=32, sampler=train_sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

# -----------------------
# 5. Model (MobileNetV2)
# -----------------------
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

# Freeze most backbone layers, fine-tune only last feature block + classifier
for param in model.features.parameters():
    param.requires_grad = False
for param in model.features[-1].parameters():
    param.requires_grad = True

model.classifier = nn.Sequential(
    nn.Linear(model.last_channel, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, num_classes),
)

model = model.to(device)

# -----------------------
# 6. Loss + optimizer + scheduler
# -----------------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


# -----------------------
# 7. Train with validation
# -----------------------
epochs = 20
best_val_acc = 0.0
best_state = None
early_stop_patience = 5
epochs_without_improve = 0

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)

    train_loss = running_loss / max(running_total, 1)
    train_acc = running_correct / max(running_total, 1)
    val_loss, val_acc = evaluate(model, val_loader)
    scheduler.step(val_loss)

    print(
        f"Epoch {epoch + 1:02d}/{epochs} | "
        f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
        f"Val Loss: {val_loss:.4f} Acc: {val_acc:.3f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improve = 0
    else:
        epochs_without_improve += 1
        if epochs_without_improve >= early_stop_patience:
            print("Early stopping triggered.")
            break

if best_state is not None:
    model.load_state_dict(best_state)

# -----------------------
# 8. Save best model
# -----------------------
save_path = Path("garbage_model.pth")
torch.save(
    {
        "model_state": model.state_dict(),
        "class_names": class_names,
        "img_size": IMG_SIZE,
        "mean": IMAGENET_MEAN,
        "std": IMAGENET_STD,
    },
    save_path,
 )

print(f"Best val acc: {best_val_acc:.3f}")
print(f"Model saved to: {save_path.resolve()}")

Dataset path: /Users/mihai/Developer/garbage-optimizer/data/dataset/archive/original
Using device: mps
Classes: ['battery', 'biological', 'cardboard', 'clothes', 'glass', 'metal', 'paper', 'plastic', 'shoes', 'trash']
Num classes: 10
Epoch 1/15 | Loss: 223.8362
Epoch 2/15 | Loss: 142.4778
Epoch 3/15 | Loss: 126.5191
Epoch 4/15 | Loss: 112.9575
Epoch 5/15 | Loss: 106.4078
Epoch 6/15 | Loss: 98.8806
Epoch 7/15 | Loss: 97.2099
Epoch 8/15 | Loss: 90.0592
Epoch 9/15 | Loss: 81.4663
Epoch 10/15 | Loss: 78.6978
Epoch 11/15 | Loss: 74.6380
Epoch 12/15 | Loss: 73.8809
Epoch 13/15 | Loss: 69.1636
Epoch 14/15 | Loss: 66.3379
Epoch 15/15 | Loss: 66.8758
Model saved as garbage_model.pth
